<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Глубокое обучение. Часть 2
# Домашнее задание по теме "Механизм внимания"

Это домашнее задание проходит в формате peer-review. Это означает, что его будут проверять ваши однокурсники. Поэтому пишите разборчивый код, добавляйте комментарии и пишите выводы после проделанной работы.

В этом задании вы будете решать задачу классификации математических задач по темам (многоклассовая классификация) с помощью Transformer.

В качестве датасета возьмем датасет математических задач по разным темам. Нам необходим следующий файл:

[Файл с классами](https://docs.google.com/spreadsheets/d/13YIbphbWc62sfa-bCh8MLQWKizaXbQK9/edit?usp=drive_link&ouid=104379615679964018037&rtpof=true&sd=true)

**Hint:** не перезаписывайте модели, которые вы получите на каждом из этапов этого дз. Они ещё понадобятся.

### Инпуты

In [156]:
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(palette='summer')

In [157]:
# !pip install accelerate -U

In [158]:
# !pip install transformers

In [159]:
# !pip install evaluate

In [160]:
# !pip install datasets

In [161]:
import transformers
from transformers import AutoModel
from datasets import load_dataset
from typing import Union
from sklearn.model_selection import train_test_split

In [162]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from typing import Union, List
import pandas as pd
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from tqdm.notebook import tqdm

In [163]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

### Задание 1 (2 балла)

Напишите кастомный класс для модели трансформера для задачи классификации, использующей в качествке backbone какую-то из моделей huggingface.

Т.е. конструктор класса должен принимать на вход название модели и подгружать её из huggingface, а затем использовать в качестве backbone (достаточно возможности использовать в качестве backbone те модели, которые упомянуты в последующих пунктах)

In [164]:
### This is just an interface example. You may change it if you want.

class TransformerClassificationModel(nn.Module):
    def __init__(self, base_transformer_model: Union[str, nn.Module], num_classes: int):
        super().__init__()
        if isinstance(base_transformer_model, str):
          self.backbone = AutoModel.from_pretrained(base_transformer_model)
        else:
          self.backbone = base_transformer_model
        self.fc = nn.Linear(self.backbone.config.hidden_size, num_classes)

    def forward(self, **kwargs):
        output = self.backbone(**kwargs)
        cls_vector = output.last_hidden_state[:, 0, :]  # shape: (batch_size, hidden_size)
        outputs = self.fc(cls_vector)
        return {"logits": outputs}

### Задание 2 (1 балл)

Напишите функцию заморозки backbone у модели (если необходимо, возвращайте из функции модель)

In [165]:
def freeze_backbone_function(model: TransformerClassificationModel):
  for param in model.backbone.parameters():
    param.requires_grad = False
  return model

### Задание 3 (2 балла)

Напишите функцию, которая будет использована для тренировки (дообучения) трансформера (TransformerClassificationModel). Функция должна поддерживать обучение с замороженным и размороженным backbone.

In [166]:
import copy
from torch.utils.tensorboard import SummaryWriter

In [167]:
def evaluate_model(model, dataloader, criterion, device, writer=None, epoch=0):
    model.eval()
    total_loss = 0
    cnt_right = 0
    cnt_total = 0

    with torch.no_grad():
        for batch in tqdm(dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = criterion(outputs['logits'], batch['labels'])
            total_loss += loss.item()

            answers = outputs['logits'].argmax(dim=-1)
            print(answers[:10], batch['labels'][:10])
            cnt_right += (answers == batch['labels']).sum().item()
            cnt_total += batch['labels'].shape[0]

    avg_loss = total_loss / len(dataloader)
    accuracy = cnt_right / cnt_total
    print(f"Epoch {epoch} | val loss: {avg_loss:.4f} | accuracy: {accuracy:.4f}")

    if writer is not None:
        writer.add_scalar('Loss/val', avg_loss, epoch)
        writer.add_scalar('Accuracy/val', accuracy, epoch)

    return avg_loss, accuracy

In [168]:
def train_transformer(transformer_model, train_dataloader, val_dataloader,
                      optimizer, criterion, device,
                      freeze_backbone=True, scheduler=None, writer=None, epochs=5):
    model = transformer_model
    model = model.to(device)
    if freeze_backbone:
        freeze_backbone_function(model)

    for epoch in tqdm(range(epochs)):
        model.train()
        total_loss = 0
        global_step = epoch * len(train_dataloader)

        for batch in tqdm(train_dataloader, leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs['logits']
            loss = criterion(logits, batch['labels'])
            #print(batch['labels'].dtype, batch['labels'].unique())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()

            total_loss += loss.item()

            if writer is not None:
                    writer.add_scalar('Loss/train_step', loss.item(), global_step)
            global_step += 1

        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch} | train loss: {avg_loss:.4f}")
        if writer is not None:
            writer.add_scalar('Loss/train_epoch', avg_loss, epoch)

        evaluate_model(model, val_dataloader, criterion, device, writer=writer, epoch=epoch)

    return model

### Подготовка датасета

In [169]:
df = pd.read_excel('content/data_problems.xlsx').drop('Unnamed: 0', axis=1, errors='ignore')
# Удаляем строки с NaN
df = df.dropna(subset=['problem_text'])

# Также удаляем пустые строки (если есть)
df = df[df['problem_text'].str.strip() != '']

In [170]:
df.head()

,problem_text,topic
0,To prove that the sum of the numbers of the ex...,number_theory
1,( b) Will the statement of the previous challe...,number_theory
2,The quadratic three-member graph with the coef...,polynoms
3,Can you draw on the surface of Rubik's cube a ...,combinatorics
4,"Dima, who came from Vrunlandia, said that ther...",graphs


In [171]:
num_classes = df['topic'].nunique()
print(df['topic'].nunique())

7


In [172]:
print(df['topic'].value_counts())

topic
number_theory    2395
combinatorics    1020
dirichlet         441
polynoms          425
graphs            384
geometry          368
invariant         235
Name: count, dtype: int64


In [173]:
print(len(df))

5268


In [174]:
le = LabelEncoder()
df['labels'] = le.fit_transform(df['topic'])

In [175]:
df.head()

,problem_text,topic,labels
0,To prove that the sum of the numbers of the ex...,number_theory,5
1,( b) Will the statement of the previous challe...,number_theory,5
2,The quadratic three-member graph with the coef...,polynoms,6
3,Can you draw on the surface of Rubik's cube a ...,combinatorics,0
4,"Dima, who came from Vrunlandia, said that ther...",graphs,3


In [176]:
class_counts = df['labels'].value_counts().sort_index()
weights = 1.0 / torch.tensor(class_counts.values, dtype=torch.float)
weights = weights / weights.sum()

In [177]:
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

In [178]:
class ProblemsDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
        self.device = device

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        tokenized = self.tokenizer(
            row['problem_text'],
            truncation=True,
            max_length=64,
            padding='max_length'
        )
        return {
            'input_ids': torch.tensor(tokenized['input_ids']),
            'attention_mask': torch.tensor(tokenized['attention_mask']),
            'labels': torch.tensor(row['labels'], dtype=torch.long)  # Изменено с torch.long
        }

### Задание 4 (1 балл)

Проверьте вашу функцию из предыдущего пункта, дообучив двумя способами
*cointegrated/rubert-tiny2* из huggingface.

In [179]:
LR = 0.0005
EPOCHS = 5
SCHEDULER_LAMBDA_PARAM = 0.96
BASE_FOLDER_PATH = 'gdrive/MyDrive'
RUBERT_MODEL_NAME = "cointegrated/rubert-tiny2"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [180]:
rubert_tiny_frozen_tokenizer = AutoTokenizer.from_pretrained(RUBERT_MODEL_NAME)
rubert_tiny_frozen_transformer_model = TransformerClassificationModel(RUBERT_MODEL_NAME, num_classes)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [181]:
train_dataset = ProblemsDataset(train_df, rubert_tiny_frozen_tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = ProblemsDataset(test_df, rubert_tiny_frozen_tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [182]:
optimizer = optim.AdamW(rubert_tiny_frozen_transformer_model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss(weight=weights.to(device))
lambda_scheduler = lambda x: SCHEDULER_LAMBDA_PARAM ** x
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda_scheduler)

In [183]:
writer_frozen = SummaryWriter(os.path.join(BASE_FOLDER_PATH, 'runs/rubert_frozen'.format(RUBERT_MODEL_NAME)))

In [184]:
rubert_tiny_finetuned_with_frozen_backbone = train_transformer(
    rubert_tiny_frozen_transformer_model, train_dataloader, test_dataloader, optimizer, criterion, device,
    freeze_backbone=True, scheduler=None, writer=writer_frozen, epochs=2
)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/149 [00:00<?, ?it/s]

Epoch 0 | train loss: 1.7132


  0%|          | 0/17 [00:00<?, ?it/s]

tensor([4, 6, 0, 3, 6, 5, 2, 2, 3, 5], device='cuda:0') tensor([5, 0, 0, 1, 0, 5, 2, 2, 5, 5], device='cuda:0')
tensor([5, 3, 0, 5, 6, 1, 2, 2, 3, 5], device='cuda:0') tensor([3, 0, 6, 0, 6, 5, 2, 2, 0, 4], device='cuda:0')
tensor([0, 3, 3, 2, 1, 0, 5, 3, 5, 2], device='cuda:0') tensor([5, 5, 5, 6, 0, 0, 5, 5, 5, 2], device='cuda:0')
tensor([4, 5, 3, 5, 0, 5, 5, 6, 5, 1], device='cuda:0') tensor([4, 5, 3, 4, 1, 5, 5, 6, 1, 0], device='cuda:0')
tensor([5, 3, 3, 5, 3, 3, 2, 4, 2, 5], device='cuda:0') tensor([5, 0, 1, 5, 0, 5, 5, 3, 5, 5], device='cuda:0')
tensor([2, 3, 6, 5, 3, 1, 0, 3, 4, 5], device='cuda:0') tensor([6, 0, 5, 5, 5, 3, 5, 3, 0, 5], device='cuda:0')
tensor([3, 6, 6, 5, 5, 5, 5, 4, 5, 5], device='cuda:0') tensor([0, 5, 6, 0, 5, 5, 0, 0, 5, 5], device='cuda:0')
tensor([0, 2, 4, 0, 6, 6, 5, 0, 0, 0], device='cuda:0') tensor([3, 0, 5, 0, 5, 6, 5, 5, 1, 0], device='cuda:0')
tensor([6, 0, 6, 3, 5, 5, 5, 5, 6, 5], device='cuda:0') tensor([5, 3, 6, 3, 5, 5, 5, 5, 5, 0], device='c

  0%|          | 0/149 [00:00<?, ?it/s]

Epoch 1 | train loss: 1.4434


  0%|          | 0/17 [00:00<?, ?it/s]

tensor([4, 6, 0, 4, 6, 5, 2, 2, 0, 5], device='cuda:0') tensor([5, 0, 0, 1, 0, 5, 2, 2, 5, 5], device='cuda:0')
tensor([1, 0, 6, 5, 6, 1, 2, 2, 3, 4], device='cuda:0') tensor([3, 0, 6, 0, 6, 5, 2, 2, 0, 4], device='cuda:0')
tensor([0, 0, 3, 6, 1, 0, 5, 3, 5, 2], device='cuda:0') tensor([5, 5, 5, 6, 0, 0, 5, 5, 5, 2], device='cuda:0')
tensor([0, 5, 3, 5, 0, 4, 5, 6, 5, 1], device='cuda:0') tensor([4, 5, 3, 4, 1, 5, 5, 6, 1, 0], device='cuda:0')
tensor([5, 1, 3, 5, 3, 4, 1, 4, 1, 5], device='cuda:0') tensor([5, 0, 1, 5, 0, 5, 5, 3, 5, 5], device='cuda:0')
tensor([2, 0, 6, 5, 3, 1, 0, 1, 4, 5], device='cuda:0') tensor([6, 0, 5, 5, 5, 3, 5, 3, 0, 5], device='cuda:0')
tensor([0, 6, 6, 0, 5, 5, 5, 4, 5, 1], device='cuda:0') tensor([0, 5, 6, 0, 5, 5, 0, 0, 5, 5], device='cuda:0')
tensor([0, 0, 4, 0, 6, 6, 5, 0, 0, 0], device='cuda:0') tensor([3, 0, 5, 0, 5, 6, 5, 5, 1, 0], device='cuda:0')
tensor([6, 0, 6, 3, 5, 5, 1, 5, 6, 5], device='cuda:0') tensor([5, 3, 6, 3, 5, 5, 5, 5, 5, 0], device='c

In [185]:
print(optimizer.param_groups[0]['lr'])

0.0005


In [186]:
rubert_tiny_unfrozen_tokenizer = AutoTokenizer.from_pretrained(RUBERT_MODEL_NAME)
rubert_tiny_unfrozen_transformer_model = TransformerClassificationModel(RUBERT_MODEL_NAME, num_classes)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [187]:
optimizer = optim.AdamW(rubert_tiny_unfrozen_transformer_model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss(weight=weights.to(device))
lambda_scheduler = lambda x: SCHEDULER_LAMBDA_PARAM ** x
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda_scheduler)

In [188]:
writer_unfrozen = SummaryWriter(os.path.join(BASE_FOLDER_PATH, 'runs/rubert_unfrozen'.format(RUBERT_MODEL_NAME)))

In [189]:
rubert_tiny_finetuned_with_unfrozen_backbone = train_transformer(
    rubert_tiny_unfrozen_transformer_model, train_dataloader, test_dataloader, optimizer, criterion, device,
    freeze_backbone=False, scheduler=None, writer=writer_unfrozen, epochs=2
)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/149 [00:00<?, ?it/s]

Epoch 0 | train loss: 1.3144


  0%|          | 0/17 [00:00<?, ?it/s]

tensor([0, 0, 3, 1, 6, 1, 2, 2, 0, 1], device='cuda:0') tensor([5, 0, 0, 1, 0, 5, 2, 2, 5, 5], device='cuda:0')
tensor([0, 0, 6, 0, 6, 0, 2, 2, 0, 4], device='cuda:0') tensor([3, 0, 6, 0, 6, 5, 2, 2, 0, 4], device='cuda:0')
tensor([1, 4, 6, 6, 1, 3, 5, 3, 1, 2], device='cuda:0') tensor([5, 5, 5, 6, 0, 0, 5, 5, 5, 2], device='cuda:0')
tensor([2, 5, 3, 4, 1, 1, 5, 4, 1, 0], device='cuda:0') tensor([4, 5, 3, 4, 1, 5, 5, 6, 1, 0], device='cuda:0')
tensor([5, 1, 0, 1, 3, 1, 1, 1, 1, 5], device='cuda:0') tensor([5, 0, 1, 5, 0, 5, 5, 3, 5, 5], device='cuda:0')
tensor([6, 0, 6, 5, 0, 1, 1, 3, 0, 5], device='cuda:0') tensor([6, 0, 5, 5, 5, 3, 5, 3, 0, 5], device='cuda:0')
tensor([1, 6, 6, 3, 5, 1, 0, 1, 1, 1], device='cuda:0') tensor([0, 5, 6, 0, 5, 5, 0, 0, 5, 5], device='cuda:0')
tensor([0, 1, 1, 0, 5, 6, 1, 1, 1, 0], device='cuda:0') tensor([3, 0, 5, 0, 5, 6, 5, 5, 1, 0], device='cuda:0')
tensor([5, 0, 6, 3, 5, 1, 1, 5, 1, 0], device='cuda:0') tensor([5, 3, 6, 3, 5, 5, 5, 5, 5, 0], device='c

  0%|          | 0/149 [00:00<?, ?it/s]

Epoch 1 | train loss: 0.9125


  0%|          | 0/17 [00:00<?, ?it/s]

tensor([3, 0, 3, 3, 6, 5, 2, 2, 3, 5], device='cuda:0') tensor([5, 0, 0, 1, 0, 5, 2, 2, 5, 5], device='cuda:0')
tensor([3, 0, 6, 3, 6, 1, 2, 2, 3, 5], device='cuda:0') tensor([3, 0, 6, 0, 6, 5, 2, 2, 0, 4], device='cuda:0')
tensor([1, 4, 3, 3, 1, 3, 5, 3, 5, 2], device='cuda:0') tensor([5, 5, 5, 6, 0, 0, 5, 5, 5, 2], device='cuda:0')
tensor([4, 5, 3, 4, 3, 4, 5, 4, 5, 3], device='cuda:0') tensor([4, 5, 3, 4, 1, 5, 5, 6, 1, 0], device='cuda:0')
tensor([5, 1, 1, 5, 3, 4, 1, 4, 4, 5], device='cuda:0') tensor([5, 0, 1, 5, 0, 5, 5, 3, 5, 5], device='cuda:0')
tensor([6, 3, 5, 5, 5, 0, 3, 3, 0, 5], device='cuda:0') tensor([6, 0, 5, 5, 5, 3, 5, 3, 0, 5], device='cuda:0')
tensor([3, 5, 6, 3, 5, 5, 0, 5, 0, 5], device='cuda:0') tensor([0, 5, 6, 0, 5, 5, 0, 0, 5, 5], device='cuda:0')
tensor([3, 5, 0, 0, 5, 5, 5, 5, 0, 0], device='cuda:0') tensor([3, 0, 5, 0, 5, 6, 5, 5, 1, 0], device='cuda:0')
tensor([5, 3, 6, 3, 5, 5, 1, 5, 0, 3], device='cuda:0') tensor([5, 3, 6, 3, 5, 5, 5, 5, 5, 0], device='c

Unfrozen лучше frozen — имхо логично, потому что при полном дообучении backbone адаптируется под задачу

### Задание 5 (1 балл)

Обучите *tbs17/MathBert* (с замороженным backbone и без заморозки), проанализируйте результаты. Сравните скоры с первым заданием. Получилось лучше или нет? Почему?

In [190]:
MATHBERT_MODEL_NAME = "tbs17/MathBert"

In [191]:
mathbert_frozen_tokenizer = AutoTokenizer.from_pretrained(MATHBERT_MODEL_NAME)
mathbert_frozen_model = TransformerClassificationModel(MATHBERT_MODEL_NAME, num_classes)
mathbert_unfrozen_model = TransformerClassificationModel(MATHBERT_MODEL_NAME, num_classes)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: tbs17/MathBert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: tbs17/MathBert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [192]:
train_dataset_math = ProblemsDataset(train_df, mathbert_frozen_tokenizer)
train_dataloader_math = DataLoader(train_dataset_math, batch_size=8, shuffle=True)
test_dataset_math = ProblemsDataset(test_df, mathbert_frozen_tokenizer)
test_dataloader_math = DataLoader(test_dataset_math, batch_size=8, shuffle=False)

In [193]:
optimizer_frozen = optim.AdamW(mathbert_frozen_model.parameters(), lr=LR)
criterion_frozen = torch.nn.CrossEntropyLoss(weight=weights.to(device))
scheduler_frozen = optim.lr_scheduler.LambdaLR(optimizer_frozen, lr_lambda=lambda x: SCHEDULER_LAMBDA_PARAM ** x)
writer_math_frozen = SummaryWriter(os.path.join(BASE_FOLDER_PATH, 'runs/mathbert_frozen'))

In [194]:
mathbert_finetuned_frozen = train_transformer(
    mathbert_frozen_model, train_dataloader_math, test_dataloader_math,
    optimizer_frozen, criterion_frozen, device,
    freeze_backbone=True, scheduler=None, writer=writer_math_frozen,epochs=2
)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/593 [00:00<?, ?it/s]

Epoch 0 | train loss: 1.5304


  0%|          | 0/66 [00:00<?, ?it/s]

tensor([0, 6, 3, 3, 6, 6, 2, 2], device='cuda:0') tensor([5, 0, 0, 1, 0, 5, 2, 2], device='cuda:0')
tensor([3, 2, 5, 3, 2, 6, 4, 3], device='cuda:0') tensor([5, 5, 5, 3, 2, 0, 5, 5], device='cuda:0')
tensor([2, 2, 3, 3, 2, 5, 4, 6], device='cuda:0') tensor([2, 2, 3, 0, 2, 5, 4, 6], device='cuda:0')
tensor([5, 6, 4, 5, 1, 6, 1, 4], device='cuda:0') tensor([5, 6, 5, 5, 0, 5, 1, 5], device='cuda:0')
tensor([0, 0, 6, 5, 6, 0, 2, 2], device='cuda:0') tensor([3, 0, 6, 0, 6, 5, 2, 2], device='cuda:0')
tensor([5, 6, 4, 5, 0, 4, 4, 6], device='cuda:0') tensor([0, 4, 4, 5, 0, 4, 5, 5], device='cuda:0')
tensor([5, 3, 1, 1, 3, 6, 3, 5], device='cuda:0') tensor([5, 3, 0, 3, 0, 5, 3, 5], device='cuda:0')
tensor([0, 1, 3, 4, 5, 5, 6, 6], device='cuda:0') tensor([0, 5, 0, 5, 5, 5, 5, 6], device='cuda:0')
tensor([4, 3, 6, 6, 3, 0, 5, 0], device='cuda:0') tensor([5, 5, 5, 6, 0, 0, 5, 5], device='cuda:0')
tensor([5, 2, 3, 6, 4, 4, 6, 0], device='cuda:0') tensor([5, 2, 3, 5, 4, 5, 2, 1], device='cuda:0')


  0%|          | 0/593 [00:00<?, ?it/s]

Epoch 1 | train loss: 1.3075


  0%|          | 0/66 [00:00<?, ?it/s]

tensor([4, 6, 3, 4, 6, 5, 2, 2], device='cuda:0') tensor([5, 0, 0, 1, 0, 5, 2, 2], device='cuda:0')
tensor([3, 2, 5, 3, 2, 6, 4, 0], device='cuda:0') tensor([5, 5, 5, 3, 2, 0, 5, 5], device='cuda:0')
tensor([2, 2, 3, 3, 2, 5, 4, 6], device='cuda:0') tensor([2, 2, 3, 0, 2, 5, 4, 6], device='cuda:0')
tensor([5, 6, 4, 5, 1, 6, 1, 4], device='cuda:0') tensor([5, 6, 5, 5, 0, 5, 1, 5], device='cuda:0')
tensor([0, 0, 6, 5, 6, 5, 2, 2], device='cuda:0') tensor([3, 0, 6, 0, 6, 5, 2, 2], device='cuda:0')
tensor([3, 3, 4, 5, 4, 4, 4, 6], device='cuda:0') tensor([0, 4, 4, 5, 0, 4, 5, 5], device='cuda:0')
tensor([5, 3, 4, 3, 3, 6, 1, 5], device='cuda:0') tensor([5, 3, 0, 3, 0, 5, 3, 5], device='cuda:0')
tensor([5, 4, 3, 4, 5, 5, 6, 6], device='cuda:0') tensor([0, 5, 0, 5, 5, 5, 5, 6], device='cuda:0')
tensor([4, 4, 6, 6, 3, 3, 5, 0], device='cuda:0') tensor([5, 5, 5, 6, 0, 0, 5, 5], device='cuda:0')
tensor([5, 2, 3, 2, 4, 4, 6, 0], device='cuda:0') tensor([5, 2, 3, 5, 4, 5, 2, 1], device='cuda:0')


In [197]:
optimizer_unfrozen = optim.AdamW(mathbert_unfrozen_model.parameters(), lr=0.05)
scheduler_unfrozen = optim.lr_scheduler.LambdaLR(optimizer_unfrozen, lr_lambda=lambda x: SCHEDULER_LAMBDA_PARAM ** x)
criterion_unfrozen = torch.nn.CrossEntropyLoss(weight=weights.to(device))
writer_math_unfrozen = SummaryWriter(os.path.join(BASE_FOLDER_PATH, 'runs/mathbert_unfrozen'))

In [198]:
mathbert_finetuned_unfrozen = train_transformer(
    mathbert_unfrozen_model, train_dataloader_math, test_dataloader_math,
    optimizer_unfrozen, criterion_unfrozen, device,
    freeze_backbone=False, scheduler=None, writer=writer_math_unfrozen, epochs=1
)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/593 [00:00<?, ?it/s]

Epoch 0 | train loss: 3.1200


  0%|          | 0/66 [00:00<?, ?it/s]

tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([5, 0, 0, 1, 0, 5, 2, 2], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([5, 5, 5, 3, 2, 0, 5, 5], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([2, 2, 3, 0, 2, 5, 4, 6], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([5, 6, 5, 5, 0, 5, 1, 5], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([3, 0, 6, 0, 6, 5, 2, 2], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([0, 4, 4, 5, 0, 4, 5, 5], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([5, 3, 0, 3, 0, 5, 3, 5], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([0, 5, 0, 5, 5, 5, 5, 6], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([5, 5, 5, 6, 0, 0, 5, 5], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0') tensor([5, 2, 3, 5, 4, 5, 2, 1], device='cuda:0')


### Задание 6 (1 балл)

Напишите функцию для отрисовки карт внимания первого слоя для моделей из задания

In [199]:
def draw_first_layer_attention_maps(attention_head_ids: List, text: str, model: TransformerClassificationModel, tokenizer):
    model.eval()

    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(next(model.parameters()).device)

    with torch.no_grad():
        output = model.backbone(**inputs, output_attentions=True)

    first_layer_attention = output.attentions[0].squeeze(0).cpu()

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze(0))

    fig, axes = plt.subplots(1, len(attention_head_ids), figsize=(6 * len(attention_head_ids), 6))
    if len(attention_head_ids) == 1:
        axes = [axes]

    for ax, head_id in zip(axes, attention_head_ids):
        attention_map = first_layer_attention[head_id].numpy()
        sns.heatmap(attention_map, xticklabels=tokens, yticklabels=tokens, ax=ax, cmap='viridis')
        ax.set_title(f'Head {head_id}')
        ax.tick_params(axis='x', rotation=90)
        ax.tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.show()

### Задание 7 (1 балл)

Проведите инференс для всех моделей **ДО ДООБУЧЕНИЯ** на 2-3 текстах из датасета. Посмотрите на головы Attention первого слоя в каждой модели на выбранных текстах (отрисуйте их отдельно).

Попробуйте их проинтерпретировать. Какие связи улавливают карты внимания? (если в модели много голов Attention, то проинтерпретируйте наиболее интересные)

In [200]:
sample_texts = df['problem_text'].iloc[:3].tolist()

In [ ]:
models_before = {
    'rubert_tiny': (rubert_tiny_frozen_transformer_model, rubert_tiny_frozen_tokenizer),
    'mathbert': (mathbert_frozen_model, mathbert_frozen_tokenizer)
}

In [202]:
for model_name, (model, tokenizer) in models_before.items():
    print(f"\n=== {model_name} ===")
    for i, text in enumerate(sample_texts):
        print(f"\nТекст {i+1}: {text[:80]}...")
        draw_first_layer_attention_maps([0, 1, 2], text, model, tokenizer)


=== rubert_tiny ===

Текст 1: To prove that the sum of the numbers of the exact square cannot be equal to 5....


IndexError: tuple index out of range

### Задание 8 (1 балл)

Сделайте то же самое для дообученных моделей. Изменились ли карты внимания и связи, которые они улавливают? Почему?

In [ ]:
models_after = {
    'rubert_tiny_frozen': (rubert_tiny_finetuned_with_frozen_backbone, rubert_tiny_frozen_tokenizer),
    'rubert_tiny_unfrozen': (rubert_tiny_finetuned_with_unfrozen_backbone, rubert_tiny_frozen_tokenizer),
    'mathbert_frozen': (mathbert_finetuned_frozen, mathbert_frozen_tokenizer),
    'mathbert_unfrozen': (mathbert_finetuned_unfrozen, mathbert_frozen_tokenizer)
}

for model_name, (model, tokenizer) in models_after.items():
    print(f"\n=== {model_name} ===")
    for i, text in enumerate(sample_texts):
        print(f"\nТекст {i+1}: {text[:80]}...")
        draw_first_layer_attention_maps([0, 1, 2], text, model, tokenizer)


=== rubert_tiny_frozen ===

Текст 1: To prove that the sum of the numbers of the exact square cannot be equal to 5....


IndexError: tuple index out of range

In [ ]:
s

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from typing import Union, List, Optional, Dict, Tuple
import copy
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Hugging Face импорты
from transformers import (
    AutoModel, 
    AutoTokenizer, 
    AutoConfig,
    get_scheduler,
    DataCollatorWithPadding
)

from datasets import Dataset as HFDataset

from datetime import datetime
from torch.utils.tensorboard import SummaryWriter
from torch.nn.utils.rnn import pad_sequence
from sklearn.preprocessing import LabelEncoder

import gdown
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

sns.set(palette='summer')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def set_seed(seed=42):
    """
    Фиксация всех источников случайности
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Детерминированные алгоритмы
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Фиксация хэшей Python
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Для CUDA
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   
    
    # Фиксация для DataLoader
    def seed_worker(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    
    return seed_worker

# Фиксация случайности
SEED = 42
seed_worker = set_seed(SEED)

# Параметты
os.makedirs('content', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)

BASE_FOLDER_PATH = 'models'
BATCH_SIZE = 16
EPOCHS = 5
MODEL_NAME = 'math_problems_classifier'
LR = 3e-5
N_HEADS = 1

MIN_LENGTH = 10
MAX_LENGTH = 128

DROPOUT = 0.3

MODEL_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODEL_NAME)
SCHEDULER_LAMBDA_PARAM = 0.95
PAD_IND = 0

os.makedirs(MODEL_FOLDER_PATH, exist_ok=True)

### Загрузка и обработка данных
file_id = '13YIbphbWc62sfa-bCh8MLQWKizaXbQK9'
output = 'content/data_problems.xlsx'

if not os.path.exists(output):
    gdown.download(id=file_id, output=output, quiet=False)

df = pd.read_excel('content/data_problems.xlsx').drop('Unnamed: 0', axis=1, errors='ignore')

def clean_text(text):
    """
    Подготовка текста
    """
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    text = ' '.join(text.split())
    return text

df_prep = df.copy()
df_prep['problem_text'] = df_prep['problem_text'].apply(clean_text)

def prepare_df(df):
    """
    Подготовка датафрейма
    """
    df = df.copy()
    df = df.dropna()
    df = df[df['problem_text'] != '']
    df = df.drop_duplicates()
    return df

df_prep = prepare_df(df_prep)

# Фильтрация по длине
df_prep = df_prep[df_prep['problem_text'].apply(len) >= MIN_LENGTH].copy()

# Преобразование меток
label_encoder = LabelEncoder()
topics = df_prep['topic'].values
label_encoder.fit(topics)
df_prep['labels'] = label_encoder.transform(df_prep['topic'])
num_classes = len(label_encoder.classes_)

# Разбиение train (80%), val (20%)
train_df, val_df = train_test_split(
    df_prep, test_size=0.2, random_state=SEED, 
    stratify=df_prep['topic']
)

print(f'Train: {len(train_df)} примеров')
print(f'Val: {len(val_df)} примеров')

### Задание 1: Класс модели
class TransformerClassificationModel(nn.Module):
    def __init__(self, base_transformer_model: Union[str, nn.Module], num_classes: int):
        super().__init__()
        if isinstance(base_transformer_model, str):
            self.backbone = AutoModel.from_pretrained(base_transformer_model)
        else:
            self.backbone = base_transformer_model
        self.fc = nn.Linear(self.backbone.config.hidden_size, num_classes)

    def forward(self, **kwargs):
        output = self.backbone(**kwargs)
        cls_vector = output.last_hidden_state[:, 0, :]
        outputs = self.fc(cls_vector)
        return {"logits": outputs}

### Задание 2: Функция заморозки backbone
def freeze_backbone_function(model: TransformerClassificationModel):
    for param in model.backbone.parameters():
        param.requires_grad = False
    return model

### Задание 3: Функция для тренировки
def evaluate_model(model, dataloader, criterion, device, writer=None, epoch=0):
    model.eval()
    total_loss = 0
    cnt_right = 0
    cnt_total = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validation'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = criterion(outputs['logits'], batch['labels'])
            total_loss += loss.item()

            answers = outputs['logits'].argmax(dim=-1)
            cnt_right += (answers == batch['labels']).sum().item()
            cnt_total += batch['labels'].shape[0]

    avg_loss = total_loss / len(dataloader)
    accuracy = cnt_right / cnt_total
    
    return avg_loss, accuracy

def train_transformer(transformer_model, train_dataloader, val_dataloader,
                      optimizer, criterion, device,
                      freeze_backbone=True, scheduler=None, writer=None, epochs=5):
    model = transformer_model
    model = model.to(device)
    if freeze_backbone:
        freeze_backbone_function(model)
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        cnt_right = 0
        cnt_total = 0
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{epochs} - Training'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs['logits']
            loss = criterion(logits, batch['labels'])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()

            total_loss += loss.item()
            answers = logits.argmax(dim=-1)
            cnt_right += (answers == batch['labels']).sum().item()
            cnt_total += batch['labels'].shape[0]

        avg_train_loss = total_loss / len(train_dataloader)
        train_acc = cnt_right / cnt_total
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        
        print(f"Epoch {epoch+1} | train loss: {avg_train_loss:.4f} | train acc: {train_acc:.4f}")
        
        val_loss, val_acc = evaluate_model(model, val_dataloader, criterion, device, writer=writer, epoch=epoch+1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1} | val loss: {val_loss:.4f} | val acc: {val_acc:.4f}")
        
        if writer is not None:
            writer.add_scalar('Loss/train_epoch', avg_train_loss, epoch+1)
            writer.add_scalar('Loss/val_epoch', val_loss, epoch+1)
            writer.add_scalar('Accuracy/train_epoch', train_acc, epoch+1)
            writer.add_scalar('Accuracy/val_epoch', val_acc, epoch+1)

    return model, history

### Вспомогательная функция для графиков
def plot_training_history(history, title='History'):
    num_epochs = len(history['train_loss'])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # График Loss
    ax1.plot(range(1, num_epochs+1), history['train_loss'], label='Train Loss', marker='o')
    ax1.plot(range(1, num_epochs+1), history['val_loss'], label='Val Loss', marker='s')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} - Losses')
    ax1.legend()
    ax1.grid(True)

    # График Accuracy
    ax2.plot(range(1, num_epochs+1), history['train_acc'], label='Train acc', marker='o')
    ax2.plot(range(1, num_epochs+1), history['val_acc'], label='Val acc', marker='s')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy Score')
    ax2.set_title(f'{title} - Accuracies')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

### Dataset класс
class ProblemsDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
        self.device = device

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        tokenized = self.tokenizer(
            row['problem_text'],
            truncation=True,
            max_length=MAX_LENGTH,
            padding='max_length'
        )
        return {
            'input_ids': torch.tensor(tokenized['input_ids']),
            'attention_mask': torch.tensor(tokenized['attention_mask']),
            'labels': torch.tensor(row['labels'], dtype=torch.long)
        }

### Задание 4: Обучение RuBERT
LR = 5e-5
RUBERT_MODEL_NAME = "cointegrated/rubert-tiny2"

# Подсчет весов для классов
class_counts = train_df['labels'].value_counts().sort_index()
weights = 1.0 / torch.tensor(class_counts.values, dtype=torch.float)
weights = weights / weights.sum()

print("\n" + "="*50)
print("Задание 4: Обучение RuBERT")
print("="*50)

# Frozen backbone
print("\n--- RuBERT с замороженным backbone ---")
rubert_tokenizer = AutoTokenizer.from_pretrained(RUBERT_MODEL_NAME)
train_dataset = ProblemsDataset(train_df, rubert_tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataset = ProblemsDataset(val_df, rubert_tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

rubert_frozen_model = TransformerClassificationModel(RUBERT_MODEL_NAME, num_classes)
optimizer = optim.AdamW(rubert_frozen_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

rubert_frozen_model, history_rubert_frozen = train_transformer(
    rubert_frozen_model, train_dataloader, val_dataloader, optimizer, criterion, device,
    freeze_backbone=True, scheduler=None, epochs=EPOCHS
)

# Unfrozen backbone
print("\n--- RuBERT без заморозки backbone ---")
rubert_unfrozen_model = TransformerClassificationModel(RUBERT_MODEL_NAME, num_classes)
optimizer = optim.AdamW(rubert_unfrozen_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

rubert_unfrozen_model, history_rubert_unfrozen = train_transformer(
    rubert_unfrozen_model, train_dataloader, val_dataloader, optimizer, criterion, device,
    freeze_backbone=False, scheduler=None, epochs=EPOCHS
)

plot_training_history(history_rubert_frozen, 'RuBERT (Backbone frozen)')
plot_training_history(history_rubert_unfrozen, 'RuBERT (Backbone full)')

print(f'Best Accuracy RuBERT frozen: {max(history_rubert_frozen["val_acc"]):.4f}')
print(f'Best Accuracy RuBERT unfrozen: {max(history_rubert_unfrozen["val_acc"]):.4f}')

### Задание 5: Обучение MathBert
print("\n" + "="*50)
print("Задание 5: Обучение MathBert")
print("="*50)

MATHBERT_MODEL_NAME = "tbs17/MathBert"

# Frozen backbone
print("\n--- MathBert с замороженным backbone ---")
mathbert_tokenizer = AutoTokenizer.from_pretrained(MATHBERT_MODEL_NAME)
train_dataset_math = ProblemsDataset(train_df, mathbert_tokenizer)
train_dataloader_math = DataLoader(train_dataset_math, batch_size=BATCH_SIZE, shuffle=True)
val_dataset_math = ProblemsDataset(val_df, mathbert_tokenizer)
val_dataloader_math = DataLoader(val_dataset_math, batch_size=BATCH_SIZE, shuffle=False)

mathbert_frozen_model = TransformerClassificationModel(MATHBERT_MODEL_NAME, num_classes)
optimizer = optim.AdamW(mathbert_frozen_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

mathbert_frozen_model, history_mathbert_frozen = train_transformer(
    mathbert_frozen_model, train_dataloader_math, val_dataloader_math, optimizer, criterion, device,
    freeze_backbone=True, scheduler=None, epochs=EPOCHS
)

# Unfrozen backbone
print("\n--- MathBert без заморозки backbone ---")
mathbert_unfrozen_model = TransformerClassificationModel(MATHBERT_MODEL_NAME, num_classes)
optimizer = optim.AdamW(mathbert_unfrozen_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

mathbert_unfrozen_model, history_mathbert_unfrozen = train_transformer(
    mathbert_unfrozen_model, train_dataloader_math, val_dataloader_math, optimizer, criterion, device,
    freeze_backbone=False, scheduler=None, epochs=EPOCHS
)

plot_training_history(history_mathbert_frozen, 'MathBert (Backbone frozen)')
plot_training_history(history_mathbert_unfrozen, 'MathBert (Backbone full)')

print(f'Best Accuracy MathBert frozen: {max(history_mathbert_frozen["val_acc"]):.4f}')
print(f'Best Accuracy MathBert unfrozen: {max(history_mathbert_unfrozen["val_acc"]):.4f}')

# Сравнение моделей
print("\n" + "="*50)
print("Сравнение результатов")
print("="*50)

models_results = {
    'RuBERT (frozen)': max(history_rubert_frozen['val_acc']),
    'RuBERT (unfrozen)': max(history_rubert_unfrozen['val_acc']),
    'MathBert (frozen)': max(history_mathbert_frozen['val_acc']),
    'MathBert (unfrozen)': max(history_mathbert_unfrozen['val_acc'])
}

for model_name, acc in models_results.items():
    print(f"{model_name}: {acc:.4f}")

# Bar plot сравнения
plt.figure(figsize=(10, 6))
bars = plt.bar(models_results.keys(), models_results.values(), color=['r', 'g', 'b', 'c'])
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Сравнение точности моделей')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')

for bar, acc in zip(bars, models_results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

### Задание 6: Функция для отрисовки карт внимания
def draw_first_layer_attention_maps(attention_head_ids: List, text: str, model: TransformerClassificationModel, tokenizer):
    model.eval()

    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(next(model.parameters()).device)

    with torch.no_grad():
        output = model.backbone(**inputs, output_attentions=True)

    first_layer_attention = output.attentions[0].squeeze(0).cpu()

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze(0))

    fig, axes = plt.subplots(1, len(attention_head_ids), figsize=(6 * len(attention_head_ids), 6))
    if len(attention_head_ids) == 1:
        axes = [axes]

    for ax, head_id in zip(axes, attention_head_ids):
        attention_map = first_layer_attention[head_id].numpy()
        sns.heatmap(attention_map, xticklabels=tokens, yticklabels=tokens, ax=ax, cmap='viridis')
        ax.set_title(f'Head {head_id}')
        ax.tick_params(axis='x', rotation=90)
        ax.tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.show()

### Задание 7: Инференс ДО дообучения
print("\n" + "="*50)
print("Задание 7: Инференс ДО дообучения")
print("="*50)

sample_texts = df_prep['problem_text'].iloc[:3].tolist()

models_before = {
    'RuBERT (before)': (TransformerClassificationModel(RUBERT_MODEL_NAME, num_classes).to(device), rubert_tokenizer),
    'MathBert (before)': (TransformerClassificationModel(MATHBERT_MODEL_NAME, num_classes).to(device), mathbert_tokenizer)
}

for model_name, (model, tokenizer) in models_before.items():
    print(f"\n=== {model_name} ===")
    for i, text in enumerate(sample_texts):
        print(f"\nТекст {i+1}: {text[:80]}...")
        draw_first_layer_attention_maps([0, 1, 2], text, model, tokenizer)

### Задание 8: Инференс ПОСЛЕ дообучения
print("\n" + "="*50)
print("Задание 8: Инференс ПОСЛЕ дообучения")
print("="*50)

models_after = {
    'RuBERT frozen (after)': (rubert_frozen_model, rubert_tokenizer),
    'RuBERT unfrozen (after)': (rubert_unfrozen_model, rubert_tokenizer),
    'MathBert frozen (after)': (mathbert_frozen_model, mathbert_tokenizer),
    'MathBert unfrozen (after)': (mathbert_unfrozen_model, mathbert_tokenizer)
}

for model_name, (model, tokenizer) in models_after.items():
    print(f"\n=== {model_name} ===")
    for i, text in enumerate(sample_texts):
        print(f"\nТекст {i+1}: {text[:80]}...")
        draw_first_layer_attention_maps([0, 1, 2], text, model, tokenizer)